In [1]:
import json
from contextlib import suppress
from datetime import datetime, timedelta
from pathlib import Path

import boto3
import numpy as np
import s3fs
import xarray as xr
from authlib.integrations.requests_client import OAuth2Session
from sentinelhub import SHConfig
from tqdm import tqdm
from pyproj import Transformer

/home/jonas/Documents/Projects/2024/change-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = SHConfig()
client = OAuth2Session(config.sh_client_id, config.sh_client_secret)
token = client.fetch_token(config.sh_token_url)

In [3]:
# All of this should move to a config file which is built on first run through.
# If rerun, it should just take from the config

region = "eu-central-1"
bucket_name = "sh-change-detection-test"
zarr_name = "2harm.zarr"

data_folder = Path("./data")
data_folder.mkdir(exist_ok=True)
out_beta = data_folder / "beta.tif"

zarr_id = "3799bb79-fb5e-4e4b-afa0-3095772b3ba9"

# Fitting harmonic regression

In [4]:
process_api = "https://services.sentinel-hub.com/api/v1/process"

def base_request(data: list, evalscript: str):
    res = 5  # meters

    aoi = [696920, 4528660, 700140, 4532020]
    crs = "http://www.opengis.net/def/crs/EPSG/0/32614"
    return {
        "input": {"bounds": {"bbox": aoi, "properties": {"crs": crs}}, "data": data},
        "output": {
            "resx": res,
            "resy": res,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": evalscript,
    }

In [5]:
with open("../beta_2harm.js", "r") as src:
    beta_evalscript = src.read()

beta_data = [
    {
        "dataFilter": {
            "timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"},
            "mosaickingOrder": "leastRecent"
            },
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
    }
]

beta_request = base_request(beta_data, beta_evalscript)

In [6]:
beta = client.post(process_api, json=beta_request)

beta.raise_for_status()

with open(out_beta, "wb") as f:
    f.write(beta.content)

## Setting up zarr

zarr will be ingested to Sentinel Hub and holds the regression coefficients, process value, and any other values needed for updating the process value i.e. RMSE.

In [7]:
# Make new Bucket

boto3.setup_default_session(profile_name="default")
s3_client = boto3.client("s3", region_name=region)
location = {"LocationConstraint": region}
with suppress(s3_client.exceptions.BucketAlreadyOwnedByYou):
    s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=location)

In [47]:
# Set SH BYOC bucket policy
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "Sentinel Hub permissions",
            "Effect": "Allow",
            "Principal": {"AWS": "arn:aws:iam::614251495211:root"},
            "Action": ["s3:GetBucketLocation", "s3:ListBucket", "s3:GetObject"],
            "Resource": [f"arn:aws:s3:::{bucket_name}", f"arn:aws:s3:::{bucket_name}/*"],
        }
    ],
}

# Convert the policy from JSON dict to string
bucket_policy = json.dumps(bucket_policy)

# Set the new policy
s3 = boto3.client("s3")
s3.put_bucket_policy(Bucket=bucket_name, Policy=bucket_policy)

{'ResponseMetadata': {'RequestId': 'Z8RV42D3N5V0D3VB',
  'HostId': 'PoAGhShhgT3975bqMr8pyx74L2plA1aIwCwvRxutJTFfeKc3IQ6RTaJkpPVGt0kep8TQHck+ZUXTtrgV46yoAw==',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'PoAGhShhgT3975bqMr8pyx74L2plA1aIwCwvRxutJTFfeKc3IQ6RTaJkpPVGt0kep8TQHck+ZUXTtrgV46yoAw==',
   'x-amz-request-id': 'Z8RV42D3N5V0D3VB',
   'date': 'Wed, 17 Apr 2024 19:12:33 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

In [13]:
beta_ds = xr.open_dataset(out_beta, band_as_variable=True)

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as 

In [14]:
order = ["y", "x"]
reordered_indexes = {index_name: beta_ds.indexes[index_name] for index_name in order}
beta_ds = beta_ds.reindex(reordered_indexes)

In [15]:
beta_ds = beta_ds.rename({"band_1": "c1", "band_2": "c2", "band_3": "c3"})
beta_ds["rmse"] = xr.zeros_like(beta_ds["c1"])
beta_ds["process"] = xr.zeros_like(beta_ds["c1"])

In [193]:
process = xr.zeros_like(beta_ds["c1"])

In [194]:
s3_out = s3fs.S3FileSystem(anon=False, profile="default")
store_out = s3fs.S3Map(root=f"s3://{bucket_name}/{zarr_name}", s3=s3_out, check=False)
process.rename("process").to_zarr(store_out, mode="a")

In [16]:
s3_out = s3fs.S3FileSystem(anon=False, profile="default")
store_out = s3fs.S3Map(root=f"s3://{bucket_name}/{zarr_name}", s3=s3_out, check=False)
beta_ds.to_zarr(store_out, mode="a")

In [115]:
# If session token needs to be refreshed:
s3fs.S3FileSystem.clear_instance_cache()

In [55]:
# Make new Zarr storage on SH
zarr_api = "https://services.sentinel-hub.com/api/v1/zarr/collections"
zarr_data = {
    "name": "CCDC",
    "s3Bucket": bucket_name,
    "path": f"{zarr_name}/",
    "crs": beta_request["input"]["bounds"]["properties"]["crs"],
}
zarr_byoc = client.post(zarr_api, json=zarr_data)
zarr_byoc.raise_for_status()
zarr_id = zarr_byoc.json()["data"]["id"]

## Getting RMSE of fitted period

In [19]:
with open("./rmse.js", "r") as src:
    sigma_evalscript = src.read()

sigma_data = [
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
        "id": "ARPS",
    },
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": f"zarr-{zarr_id}",
        "id": "beta",
    },
]

sigma_request = base_request(sigma_data, sigma_evalscript)

sigma = client.post(process_api, json=sigma_request)
sigma.raise_for_status()

In [20]:
out_sigma = data_folder / "rmse.tif"

with open(out_sigma, "wb") as f:
    f.write(sigma.content)

In [128]:
sigma_ds = xr.open_dataset(out_sigma, band_as_variable=True)
beta_ds["sigma"] = sigma_ds["band_1"]

In [129]:
# This updates the sigma on cloud storage
beta_ds.to_zarr(store_out, mode="a")

# Monitor

## Test of monitoring multiple dates

In [34]:
# Updating process values
start = "2022-01-01"
end = "2023-01-01"
with open("../evalscripts/predict_ccdc.cjs", "r") as src:
    monitor_evalscript = src.read().split("// DISCARD FROM HERE", 1)[0]

# Test with updating process value on a single date

monitor_data = [
    {
        "dataFilter": {
            "timeRange": {"from": f"{start}T00:00:00Z", "to": f"{end}T23:59:59Z"},
            "mosaickingOrder": "leastRecent"
            },
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
        "id": "ARPS",
    },
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": f"zarr-{zarr_id}",
        "id": "beta",
    },
]

monitor_request = base_request(monitor_data, monitor_evalscript)
monitor = client.post(process_api, json=monitor_request)
monitor.raise_for_status()

out_monitor = data_folder / f"sens5bound10.tif"

with open(out_monitor, "wb") as f:
    f.write(monitor.content)

In [32]:
# Updating process values
date = "2022-07-20"
with open("predict_single_date.js", "r") as src:
    monitor_evalscript = src.read()

monitor_evalscript = monitor_evalscript.replace("INSERT_DATE", date)
# Test with updating process value on a single date

monitor_data = [
    {
        "dataFilter": {"timeRange": {"from": f"{date}T00:00:00Z", "to": f"{date}T23:59:59Z"}},
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
        "id": "ARPS",
    },
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": f"zarr-{zarr_id}",
        "id": "beta",
    },
]

monitor_request = base_request(monitor_data, monitor_evalscript)
monitor = client.post(process_api, json=monitor_request)
monitor.raise_for_status()

out_monitor = data_folder / f"{date}.tif"

with open(out_monitor, "wb") as f:
    f.write(monitor.content)

## Active monitoring

This loops through dates and updates the process value on the bucket after each time step.

In [120]:
def update_process(date):
    # Updating process values
    with open("predict_single_date.js", "r") as src:
        monitor_evalscript = src.read()

    monitor_evalscript = monitor_evalscript.replace("INSERT_DATE", date)
    # Test with updating process value on a single date

    monitor_data = [
        {
            "dataFilter": {"timeRange": {"from": f"{date}T00:00:00Z", "to": f"{date}T23:59:59Z"}},
            "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
            "id": "ARPS",
        },
        {
            "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
            "type": f"zarr-{zarr_id}",
            "id": "beta",
        },
    ]

    monitor_request = base_request(monitor_data, monitor_evalscript)
    monitor = client.post(process_api, json=monitor_request)
    monitor.raise_for_status()

    out_monitor = data_folder / f"{date}.tif"

    with open(out_monitor, "wb") as f:
        f.write(monitor.content)

    s3_out = s3fs.S3FileSystem(anon=False, profile="default")
    store_out = s3fs.S3Map(root=f"s3://{bucket_name}/{zarr_name}", s3=s3_out, check=False)
    monitor_ds = xr.open_dataset(out_monitor, band_as_variable=True)
    monitor_ds.rename({"band_1": "process"}).to_zarr(store_out, mode="a")

In [32]:
d0 = datetime(2022, 1, 1)
d1 = datetime(2023, 1, 1)
dt = timedelta(days=1)
dates = np.arange(d0, d1, dt).astype(datetime)

In [53]:
xmin, ymin, xmax, ymax = [696920, 4528660, 700140, 4532020]
transformer = Transformer.from_crs("EPSG:32614", "EPSG:4326")

ul = transformer.transform(xmin,ymax)
lr = transformer.transform(xmax,ymin)

aoi_wgs = [ul[0], lr[1], ul[1], lr[0]]

In [77]:
get_available = {
  "collections": [
    f"byoc-b690a8ba-05c4-49dc-91c7-8484a1007176"
  ],
  "datetime": f"{d0.isoformat()}Z/{d1.isoformat()}Z",
  "bbox": aoi_wgs,
  "limit": 100,
  "fields": {"include": ["properties.datetime"]},
  "next": 200
}

In [110]:
dates = []
start = 0
get_available = {
        "collections": [
            f"byoc-b690a8ba-05c4-49dc-91c7-8484a1007176"
        ],
        "datetime": f"{d0.isoformat()}Z/{d1.isoformat()}Z",
        "bbox": aoi_wgs,
        "limit": 100,
        "fields": {"include": ["properties.datetime"]},
    }
while True:
    available = client.post("https://services.sentinel-hub.com/api/v1/catalog/1.0.0/search", json=get_available)
    available.raise_for_status()
    result = available.json()
    i_dates = [datetime.fromisoformat(feature["properties"]["datetime"][:-1]) for feature in result["features"]]
    dates.extend(i_dates)
    if result["context"]["returned"]<100:
        break
    start += 100
    get_available["next"] = start
dates.sort()

In [196]:
for date in tqdm(dates):
    update_process(date.strftime("%Y-%m-%d"))

 73%|███████▎  | 192/262 [21:21<07:47,  6.67s/it]


ContainsArrayError: path 'c4' contains an array